# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boddulavinaykumar6-stack/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This project is a **Ranking / Scoring** machine learning task.

The goal is to help content editors decide which content pages should be refreshed first. Instead of predicting a simple yes/no outcome, the model will assign each page a refresh priority score so that pages can be ranked from highest to lowest priority.

Ranking is appropriate because editors usually have limited time and resources. A ranked list helps them focus on the pages that are most likely to benefit from review first, making the output directly useful for decision support.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the starter dataset
df = pd.read_csv("content_refresh_anonymized.csv")

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nContent types:")
print(df["content_type"].value_counts())

Rows: 30000
Columns: 44

Content types:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. Target or proxy

The desired output is a **refresh priority score** for each content page.

The starter dataset does not contain a direct "refresh priority" label, so this is currently a **proxy target** rather than an observed outcome. In a production system, the target would ideally be based on observed future performance after a later time window, such as whether refreshing similar pages improved search performance or engagement.

For this framing exercise, the goal is to assign a score that ranks pages from highest to lowest refresh priority. This score is intended to support editorial decisions rather than make causal claims.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
target_related_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate"
]

display(df[target_related_columns].head())

print("\nDoes a 'refresh_priority_score' column already exist?")
print("refresh_priority_score" in df.columns)

,content_id,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate
0,content_304f48230142,3803,29,0.76,10.6,5.88
1,content_a1fb4e703a9e,15320,7,0.05,20.3,0.00
2,content_9aa793d4d895,12581,11,0.09,36.5,0.00
3,content_331d6c4de07b,11751,58,0.49,6.2,1.28
4,content_d99b7a2d90ca,19140,24,0.13,44.0,0.00



Does a 'refresh_priority_score' column already exist?
False


## 3. Success metric

The primary success metric for this ranking task is **Precision@K**.

Precision@K measures how many of the top-ranked pages are actually good candidates for content refresh. This metric is appropriate because content editors usually have limited time and focus on reviewing only the highest-priority pages.

A higher Precision@K means the model is placing the most valuable refresh opportunities near the top of the ranked list, making it more useful for editorial decision support.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Total content pages: {len(df):,}")

# Example number of pages an editor might review first
k = 10

print(f"\nExample: Precision@{k} would evaluate only the top {k} ranked pages.")

Total content pages: 30,000

Example: Precision@10 would evaluate only the top 10 ranked pages.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content page (content item)**.

Each row in the starter dataset represents one content page summarized over a trailing 90-day period. The columns contain search performance, engagement, content, and freshness metrics for that page.

This matches the goal of the project because the model will eventually assign a refresh priority score to each individual content page.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display dataset dimensions
print(f"Dataset shape: {df.shape}")

# Display a few rows
display(df.head(3))

print("\nUnit of analysis:")
print("One row = One pseudonymized content page")

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9



Unit of analysis:
One row = One pseudonymized content page


## 5. Why ML beats a fixed rule here

A fixed rule based on a single metric is unlikely to identify the best pages for refresh because refresh priority depends on multiple interacting factors.

For example, pages with similar CTR values may have very different search volume, average position, engagement, freshness, or content age. Considering only one metric could either waste editor time or overlook valuable refresh opportunities.

A machine learning ranking model can combine many signals to estimate refresh priority more effectively than a single if-statement. The goal is to provide decision support by identifying patterns in historical data rather than relying on one manually defined rule.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Display a few signals that could contribute to refresh priority
signals = [
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "days_since_last_update"
]

display(df[signals].head())

print(f"\nNumber of signals shown: {len(signals)}")
print("These signals illustrate why a single rule is unlikely to capture refresh priority.")

,search_volume,ctr,avg_position,engagement_rate,days_since_last_update
0,10.0,0.76,10.6,5.88,20
1,90.0,0.05,20.3,0.00,25
2,0.0,0.09,36.5,0.00,20
3,10.0,0.49,6.2,1.28,22
4,0.0,0.13,44.0,0.00,14



Number of signals shown: 5
These signals illustrate why a single rule is unlikely to capture refresh priority.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.